# C12-classical-models — Session 5: Voting, Bagging, Random Forests, and Boosting

*One 90-minute session. Every randomized estimator and resampling operation uses seed 20260804.*


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

SEED = 20260804
ATOL = 1e-10
RTOL = 1e-8
rng = np.random.default_rng(SEED)


## 1. Hard voting, soft voting, and additive scores

An ensemble combines base estimators. **Hard voting** chooses the majority predicted class, with
the smaller class label on an exact tie in this unit. **Soft voting** averages class-probability
vectors and takes the largest mean probability; it requires comparable probability estimates.
Boosting instead sums weighted base-learner votes/scores. These three aggregation rules can give
different answers on the same base predictions.

**Checkpoint 1A.** Hard votes `(0,1,1)` predict which class?

**Checkpoint 1B.** Why can one poorly calibrated member distort soft voting?


## 2. Bootstrap aggregation and variance reduction

A bootstrap sample of size $N$ draws $N$ training indices **with replacement**. Each bagged
estimator fits one sample; classification predictions are aggregated. Averaging $B$ estimators
with variance $\sigma^2$ and pairwise correlation $\rho$ has variance
$\rho\sigma^2+(1-\rho)\sigma^2/B$ under the equal-variance model. More estimators reduce only the
uncorrelated component, so diversity matters.

Out-of-bag rows for one estimator are those never sampled by it. OOB scoring can provide an
internal training diagnostic, but a final held-out test remains separate.

**Checkpoint 2A.** Must a bootstrap sample contain every original row?

**Checkpoint 2B.** What variance term remains as $B\to\infty$?


In [ ]:
indices = rng.integers(0, 8, size=8)
oob = np.setdiff1d(np.arange(8), np.unique(indices))
print("bootstrap", indices, "| out of bag", oob)
assert indices.shape == (8,) and indices.dtype == np.int64


## 3. Random forests decorrelate trees

A random forest combines bootstrap sampling with random feature subsets considered at each split.
Restricting candidate features makes trees less alike, lowering correlation and potentially
improving the ensemble variance. `n_estimators` controls tree count, `max_features` feature
subsampling, and tree depth/leaf parameters control each member. `random_state` pins both samples
and feature choices.

Bagging a fixed full-feature tree and a random forest are not synonyms: the forest adds per-split
feature randomness. Neither guarantees unbiased probabilities.

**Checkpoint 3A.** Which mechanism distinguishes a random forest from plain tree bagging?

**Checkpoint 3B.** Why can lower tree correlation help even when individual trees become slightly
weaker?


## 4. AdaBoost: sequential reweighting

Initialize row weights $q_i^{(1)}=1/N$. At round $m$, fit weak classifier $h_m(x)\in\{-1,+1\}$
using these weights. Its weighted error is
$\varepsilon_m=\sum_iq_i^{(m)}\mathbf1[h_m(x_i)\ne t_i]$. For
$0<\varepsilon_m<1/2$, define

$$\alpha_m=\frac12\log\frac{1-\varepsilon_m}{\varepsilon_m},\qquad
q_i^{(m+1)}=\frac{q_i^{(m)}e^{-\alpha_mt_ih_m(x_i)}}{Z_m},$$

where $Z_m$ normalizes weights to sum to one. Misclassified rows have $t_ih_m=-1$ and are
upweighted; correct rows are downweighted. The final sign is
$\operatorname{sign}(\sum_m\alpha_mh_m(x))$, with exact zero assigned $+1$ here.

**Checkpoint 4A.** Is $\alpha_m$ positive when error is below $1/2$?

**Checkpoint 4B.** Which rows receive the factor $e^{+\alpha_m}$?


In [ ]:
q = np.full(4, 0.25)
t = np.array([-1, -1, 1, 1])
h = np.array([-1, 1, 1, 1])
error = float(q[h != t].sum())
alpha = 0.5 * np.log((1.0 - error) / error)
q_next = q * np.exp(-alpha * t * h)
q_next /= q_next.sum()
assert np.isclose(error, 0.25, atol=ATOL, rtol=RTOL)
assert np.isclose(q_next.sum(), 1.0, atol=ATOL, rtol=RTOL)
assert q_next[1] > q_next[0]
print("error", error, "alpha", alpha, "weights", q_next)


## 5. Worked comparison: fit bagging, forest, and boosting

The split is fixed and stratified. Each estimator pins its seed. A state audit checks member
count, finite predictions, and held-out accuracy. `BaggingClassifier.estimators_` and
`AdaBoostClassifier.estimators_` expose fitted members; a forest also has `estimators_`.
Training-set advantage is not the selection target: compare cross-validation or validation
behavior under the same split.

**Checkpoint 5A.** Which trailing-underscore attribute exposes fitted ensemble members?

**Checkpoint 5B.** Why must all candidates use the same split or folds in a comparison?


In [ ]:
X, y = make_classification(n_samples=180, n_features=6, n_informative=4,
                           n_redundant=0, class_sep=0.8, random_state=SEED)
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3,
                                      random_state=SEED, stratify=y)
stump = DecisionTreeClassifier(max_depth=2, random_state=SEED)
models = {
    "bagging": BaggingClassifier(estimator=stump, n_estimators=15, random_state=SEED),
    "forest": RandomForestClassifier(n_estimators=20, max_depth=4,
                                     max_features="sqrt", random_state=SEED),
    "boosting": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,
                                  random_state=SEED), n_estimators=15,
                                  random_state=SEED),
}
for name, fitted in models.items():
    fitted.fit(Xtr, ytr)
    pred = fitted.predict(Xva)
    assert pred.shape == yva.shape
    print(name, len(fitted.estimators_), accuracy_score(yva, pred))

forest_member_predictions = np.vstack([
    member.predict(Xva) for member in models["forest"].estimators_
])
forest_importances = models["forest"].feature_importances_
assert forest_member_predictions.shape == (20, Xva.shape[0])
assert forest_importances.shape == (Xtr.shape[1],)
assert np.isclose(forest_importances.sum(), 1.0, atol=ATOL, rtol=RTOL)


## 6. Bagging versus boosting through bias and variance

Bagging trains members independently (parallel in concept) on resampled data and mainly targets
variance. Random forests additionally decorrelate trees. Boosting trains sequentially, changes
the row emphasis after each round, and can reduce bias by building an additive boundary; it may
be sensitive to noisy labels and outliers that keep attracting weight.

Voting is a combination rule, not a sampling or sequential-training mechanism. Choose among them
by base-model instability, desired diversity, noise, probability needs, compute, and validation
evidence.

**Checkpoint 6A.** Which method changes training weights using earlier mistakes?

**Checkpoint 6B.** Which method can fit all members independently once samples are drawn?


## 7. Comparison axes, pitfalls, and exam connections

Tree ensembles remain supervised and nonlinear. Forests/bagging average high-variance rules;
boosting optimizes an additive sequence. Scaling is usually unnecessary for tree members.
Interpretability falls relative to one small tree, while predictive stability often improves.
Probability averaging is available but calibration must be checked. Validate ensemble type and
hyperparameters with fixed folds; keep the test set untouched.

**Pitfalls.** Sampling without replacement is not a bootstrap; averaging labels is not always
soft voting; omitting AdaBoost normalization breaks the next distribution; reusing unchanged
weights turns boosting into repeated fitting; and setting seeds on only the outer estimator may
not pin a custom stochastic base learner.

**Exam connection.** Expect an exact one- or two-round weight ledger or a scenario distinguishing
parallel bagging from sequential boosting. **Going deeper.** Session 6 adds unsupervised k-means
and completes the model-selection matrix.

**Checkpoint 7A.** Why can a forest be less directly interpretable than one depth-2 tree?

**Checkpoint 7B.** Name two state checks beyond final accuracy for a fitted ensemble.


## Checkpoint answers

**1A.** 1. **1B.** Its overconfident probabilities can dominate the average.

**2A.** No; replacement permits repeats and omissions. **2B.** $\rho\sigma^2$.

**3A.** Random feature subsets at each split. **3B.** Averaging benefits from diversity; reduced
correlation can outweigh modest individual weakness.

**4A.** Yes. **4B.** Misclassified rows.

**5A.** `estimators_`. **5B.** Otherwise data difficulty, not model choice, can explain score
differences.

**6A.** Boosting. **6B.** Bagging/random forests.

**7A.** It aggregates many different rule sets rather than exposing one short path. **7B.** Fitted
member count and nontrivial member/sample or feature diversity (also seeded reproducibility).
